# pyquick — Use Case 1: Hydrogenation Reaction Energy

$$\mathrm{C_2H_4 + H_2 \rightarrow C_2H_6}$$

**Workflow:** SMILES strings → 3D geometries (RDKit) → B3LYP/6-31G* calculations (`PyQuick`) →
energies, energy decomposition, atomic charges, and forces (`Result`).

In [1]:
# one-time setup: RDKit for geometry generation, h5py for the dataset section
import sys
!{sys.executable} -m pip install --quiet rdkit h5py

DEPRECATION: Configuring installation scheme with distutils config files is deprecated and will no longer work in the near future. If you are using a Homebrew or Linuxbrew Python, please see discussion at https://github.com/Homebrew/homebrew-core/issues/76621


DEPRECATION: Configuring installation scheme with distutils config files is deprecated and will no longer work in the near future. If you are using a Homebrew or Linuxbrew Python, please see discussion at https://github.com/Homebrew/homebrew-core/issues/76621

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python3.9 -m pip install --upgrade pip


In [2]:
import pyquick
from rdkit import Chem
from rdkit.Chem import AllChem

## 1. Molecules from SMILES

In [3]:
def geom_from_smiles(smiles):
    """SMILES -> (geometry string, electron count)."""
    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)                           # QC needs explicit hydrogens
    AllChem.EmbedMolecule(mol, AllChem.ETKDGv3())   # generate a 3D conformer
    AllChem.MMFFOptimizeMolecule(mol)               # quick force-field cleanup
    conf = mol.GetConformer()
    lines = []
    for atom in mol.GetAtoms():
        pos = conf.GetAtomPosition(atom.GetIdx())
        lines.append(f"{atom.GetSymbol()}  {pos.x:.6f}  {pos.y:.6f}  {pos.z:.6f}")
    nelec = sum(atom.GetAtomicNum() for atom in mol.GetAtoms())
    return '\n'.join(lines), nelec


molecules = {
    'H2':   '[H][H]',
    'C2H4': 'C=C',
    'C2H6': 'CC',
}

geometries = {name: geom_from_smiles(smi) for name, smi in molecules.items()}
print(geometries['C2H4'][0])

C  0.638964  0.191424  0.034109
C  -0.638964  -0.191424  -0.034109
H  1.121938  0.362245  0.991109
H  1.227260  0.341540  -0.865705
H  -1.227260  -0.341541  0.865706
H  -1.121937  -0.362245  -0.991109


## 2. Calculation for mutiple molecules

In [4]:
calc = pyquick.PyQuick(
    method='B3LYP', # 
    basis='6-31G*',
    keywords={'cutoff': '1.0e-9', 'denserms': '1.0e-6'},
)

results = {}
for name, (geom, nelec) in geometries.items():
    results[name] = calc.get_energy(geom, name=name)
    r = results[name]
    print(f"{name:}  E = {results[name].total_energy:} Hartree")

H2  E = -1.173994092112482 Hartree
C2H4  E = -78.58732876415365 Hartree
C2H6  E = -79.82994376134315 Hartree


In [5]:
calc = pyquick.PyQuick(
    method='B3LYP',
    basis='6-31G*',
    properties=['mulliken_charges'],   # needed by save_result() in section 7
)
# DL-Find (the default optimizer) needs >= 3 atoms; H2 uses the Cartesian optimizer
calc_lopt = pyquick.PyQuick(
    method='B3LYP',
    basis='6-31G*',
    properties=['mulliken_charges'],
    keywords={'LOPT': None},
)

results = {}
for name, (geom, nelec) in geometries.items():
    opt_calc = calc_lopt if name == 'H2' else calc
    results[name] = opt_calc.geo_opt(geom, name=name)
    r = results[name]
    flag = "converged" if r.converged else "NOT CONVERGED"
    print(f"{name:}  E = {r.total_energy:} Hartree  ({flag})")
    print(f"{name:}  optimized_coord = {r.optimized_coordinates} ")

H2  E = -1.175483571247479 Hartree  (converged)
H2  optimized_coord = [[ 3.71383475e-01 -8.20102818e-09 -8.20102844e-09]
 [-3.71383475e-01  8.20102813e-09  8.20102878e-09]] 
C2H4  E = -78.58748080802462 Hartree  (converged)
C2H4  optimized_coord = [[ 0.63668591  0.19071455  0.03388198]
 [-0.63668281 -0.19071415 -0.03388304]
 [ 1.13364381  0.3654559   0.98517622]
 [ 1.2380075   0.34461118 -0.8589037 ]
 [-1.23800918 -0.34461219  0.85890643]
 [-1.13364423 -0.3654563  -0.9851769 ]] 
C2H6  E = -79.83040514039024 Hartree  (converged)
C2H6  optimized_coord = [[-0.75845486 -0.09978167  0.01968472]
 [ 0.75845082  0.09977872 -0.01967568]
 [-1.25178852  0.700533    0.58341912]
 [-1.1863212  -0.10497551 -0.98996592]
 [-1.02332422 -1.05083304  0.49554698]
 [ 1.02332739  1.05083806 -0.49555126]
 [ 1.1863217   0.10497425  0.9899643 ]
 [ 1.25178989 -0.70053281 -0.58342226]] 


## 3. Reaction energy

In [6]:
delta_e = (results['C2H6'].total_energy
           - results['C2H4'].total_energy
           - results['H2'].total_energy)

print(f"dE(C2H4 + H2 -> C2H6) = {delta_e:} Hartree")
# to kj/mol
print(f" = {delta_e * 2625.4996} kJ/mol")

#  -137 kJ/mol

dE(C2H4 + H2 -> C2H6) = -0.06744076111813913 Hartree
 = -177.06569133936983 kJ/mol


## 4. Energy decomposition

Every `Result` carries the full energy breakdown

In [7]:
components = ['total_energy', 'nuclear_repulsion', 'e_electronic',
              'e_one_electron', 'e_two_electron', 'e_xc']

header = f"{'component':20s}" + ''.join(f"{n:>18s}" for n in results)
print(header)
print('-' * len(header))
for comp in components:
    row = f"{comp:20s}"
    for r in results.values():
        row += f"{getattr(r, comp):18.9f}"
    print(row)

component                           H2              C2H4              C2H6
--------------------------------------------------------------------------
total_energy              -1.175483571     -78.587480808     -79.830405140
nuclear_repulsion          0.712448625      33.333813052      42.108977147
e_electronic              -1.887932196    -111.921293860    -121.939382287
e_one_electron            -2.493577159    -170.283021459    -189.049138308
e_two_electron             1.178222607      68.309425957      77.699125337
e_xc                      -0.572577644      -9.947698358     -10.589369317


## 6. Forces

Call `get_grad()` for the nuclear gradient. The `Result` carries **both** the energy and the gradient (dE/dR, Hartree/Bohr)

In [8]:
import math

# gradient (forces) on ethylene
c2h4_geom = geometries['C2H4'][0]
symbols = [line.split()[0] for line in c2h4_geom.splitlines()]
grad = calc.get_grad(c2h4_geom)
print(f"total energy : {grad.total_energy} Hartree")
print(f"gradient (Hartree/Bohr), shape {grad.gradient.shape}")
for sym, g in zip(symbols, grad.gradient):
    print(f"  {sym:>2s}  {g[0]:+.10f}  {g[1]:+.10f}  {g[2]:+.10f}")

total energy : -78.58732887164354 Hartree
gradient (Hartree/Bohr), shape (6, 3)
   C  +0.0083270384  +0.0024775019  +0.0004633330
   C  -0.0083268822  -0.0024773252  -0.0004641271
   H  -0.0023391156  -0.0006799866  +0.0001783925
   H  -0.0022747049  -0.0006564602  -0.0004102929
   H  +0.0022744013  +0.0006562943  +0.0004109345
   H  +0.0023392631  +0.0006799758  -0.0001782401


## 7. Building a dataset by using `h5py`

In [9]:
import os, h5py
os.makedirs('output', exist_ok=True)

def save_result(path, r, calc):
    """adapt the schema/compression to your needs."""
    with h5py.File(path, 'w') as f:
        f['energy/total']             = r.total_energy
        f['energy/nuclear_repulsion'] = r.nuclear_repulsion
        f['energy/electronic']        = r.e_electronic
        f['charges/mulliken']         = r.mulliken
        f['geometry/atomic_numbers']  = r.atomic_numbers
        f['geometry/coordinates']     = r.coordinates          # Angstrom
        f.attrs['method'] = calc.method
        f.attrs['basis']  = calc.basis

for name, r in results.items():
    path = f'output/{name}.h5'
    save_result(path, r, calc)
    print(f'{path}  written')

output/H2.h5  written
output/C2H4.h5  written
output/C2H6.h5  written


In [10]:
with h5py.File('output/C2H4.h5', 'r') as f:
    print('  total energy   =', f['energy/total'][()])
    print('  atomic numbers =', f['geometry/atomic_numbers'][()])
    print('  coordinates =', f['geometry/coordinates'][()])

  total energy   = -78.58748080802462
  atomic numbers = [6 6 1 1 1 1]
  coordinates = [[ 0.638964  0.191424  0.034109]
 [-0.638964 -0.191424 -0.034109]
 [ 1.121938  0.362245  0.991109]
 [ 1.22726   0.34154  -0.865705]
 [-1.22726  -0.341541  0.865706]
 [-1.121937 -0.362245 -0.991109]]


## 8. error handling

In [11]:
energy_only = pyquick.PyQuick(method='HF', basis='STO-3G').get_energy(geometries['H2'][0])

for attr in ('mulliken', 'e_dispersion'):
    try:
        getattr(energy_only, attr)
    except AttributeError as e:
        print(f"{attr}:\n  {e}\n")

mulliken:
  'mulliken' is unavailable: 'mulliken_charges' was not requested for this calculation — add 'mulliken_charges' to PyQuick.properties

e_dispersion:
  'e_dispersion' is unavailable: no dispersion correction was requested for this calculation — add a dispersion keyword, e.g. PyQuick(..., keywords={'D3': None})



In [12]:
import sys, time
import pyquick

# QUICK reports input errors (e.g. a basis set name that doesn't exist) by
# calling exit(), which kills the Jupyter kernel instead of raising.
# Set RUN_CRASH_DEMO = True to see it -- then restart the kernel.
RUN_CRASH_DEMO = False

GEOM = """
    O  -0.33840   0.00380   0.23923
    H  -0.33510  -0.00190  -0.83277
    H   0.67350  -0.00190   0.59353
"""

if RUN_CRASH_DEMO:
    print("before — kernel alive", flush=True)   # flush: push it out of the buffer now
    time.sleep(0.3)                               # let ipykernel actually transmit it
    pyquick.PyQuick(method='HF', basis='NO-SUCH-BASIS').get_energy(GEOM)  # exit() here
    print("after — never reached")
else:
    print("crash demo skipped (set RUN_CRASH_DEMO = True to run it)")

crash demo skipped (set RUN_CRASH_DEMO = True to run it)
